In [ ]:
from topic_model import load_model
from db import load_articles
import pandas as pd

topic_model = load_model()

# ── 1. High-level summary ─────────────────────────────────────────────────────
# Shows topic id, document count, and the auto-generated label
topic_info = topic_model.get_topic_info()
print(topic_info[["Topic", "Count", "Name"]].to_string())

# ── 2. Words + scores for a single topic ─────────────────────────────────────
# Returns list of (word, c-TF-IDF score) tuples, best words first
topic_model.get_topic(0)  # swap 0 for any topic id

# ── 3. All topics at once ─────────────────────────────────────────────────────
all_topics = topic_model.get_topics()  # dict: {topic_id: [(word, score), ...]}

# Flatten into a readable dataframe
rows = []
for topic_id, words in all_topics.items():
    if topic_id == -1:
        continue  # skip outlier bucket
    for rank, (word, score) in enumerate(words):
        rows.append(
            {
                "topic_id": topic_id,
                "rank": rank + 1,
                "word": word,
                "score": round(score, 4),
            }
        )

df_words = pd.DataFrame(rows)
print(df_words.head(40).to_string())

# ── 4. Filter to a specific topic ─────────────────────────────────────────────
print(df_words[df_words["topic_id"] == 5])

# ── 5. Top word only per topic (good for a quick label scan) ──────────────────
top_words = df_words[df_words["rank"] == 1].sort_values("topic_id")[
    ["topic_id", "word", "score"]
]
print(top_words.to_string())

2026-04-11 17:50:44,192 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


    Topic  Count                                                          Name
0      -1   8705                       -1_minister_labour_government_political
1       0  13766                     0_prime minister_minister_australian_news
2       1    926                    1_premier league_liverpool_arsenal_chelsea
3       2    136                 2_olympic_olympics_gold medal_winter olympics
4       3    110                              3_insomnia_sleep_waking_sleeping
5       4     80         4_christmas trees_christmas tree_artificial tree_tree
6       5     47                    5_mozart_classical music_composers_baroque
7       6     36                          6_schumacher_ferrari_grand prix_prix
8       7     34                          7_nba_nba finals_basketball_gambling
9       8     34                             8_lip balm_skincare_lotion_makeup
10      9     32              9_space exploration_space race_nasa_moon landing
11     10     31                                    

In [ ]:
df_viewtopics = con.execute(
    """
    SELECT a.*, t.topic_id, t.topic_label, t.topic_prob
    FROM cleaned_articles a
    LEFT JOIN article_topics t ON a.id = t.id
    """
).fetchdf()

print(df_viewtopics)